In [2]:
with open('orders.csv','w') as f:
    f.write('customer_id,customer_name,delivery_date,status,issue\n')
    f.write('1,vaishali,2026-05-01,delayed,late delivery\n')
    f.write('2,rahul,2026-05-12,delivered,no issue\n')
    f.write('3,priya,2026-05-03,delayed,damaged package\n')
    f.write('4,arun,2026-05-08,delayed,late delivery\n')


In [3]:
import pandas as pd

df=pd.read_csv('orders.csv')

print(df)

   customer_id customer_name delivery_date     status            issue
0            1      vaishali    2026-05-01    delayed    late delivery
1            2         rahul    2026-05-12  delivered         no issue
2            3         priya    2026-05-03    delayed  damaged package
3            4          arun    2026-05-08    delayed    late delivery


# **Week - 2**

In [4]:
import numpy as np
df.dropna(inplace=True)
df['delivery_date']=pd.to_datetime(df['delivery_date'])
df['delay_days']=(pd.Timestamp.today()-df['delivery_date']).dt.days
df['delayed']=np.where(df['delay_days']>5,1,0)
print(df)
print(df.groupby('customer_name')['delayed'].sum().sort_values(ascending=False))
print(df['issue'].value_counts())

   customer_id customer_name delivery_date     status            issue  \
0            1      vaishali    2026-05-01    delayed    late delivery   
1            2         rahul    2026-05-12  delivered         no issue   
2            3         priya    2026-05-03    delayed  damaged package   
3            4          arun    2026-05-08    delayed    late delivery   

   delay_days  delayed  
0          13        1  
1           2        0  
2          11        1  
3           6        1  
customer_name
arun        1
priya       1
vaishali    1
rahul       0
Name: delayed, dtype: int64
issue
late delivery      2
no issue           1
damaged package    1
Name: count, dtype: int64


# **Week -3**

In [5]:
!pip install pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col,count
spark=SparkSession.builder.appName('customer_orders').getOrCreate()

In [6]:
customers=spark.createDataFrame([
(1,'vaishali','chennai'),
(2,'rahul','bangalore'),
(3,'priya','hyderabad'),
(4,'arun','chennai')],
['customer_id','customer_name','region'])

In [7]:
orders=spark.createDataFrame([
(101,1,'delayed'),
(102,2,'delivered'),
(103,3,'delayed'),
(104,4,'delayed')
],
['order_id','customer_id','status']
)

In [10]:
joined=orders.join(
customers,
'customer_id'
)

joined.show()
result=joined.filter(
col('status')=='delayed'
).groupBy(
'region'
).agg(
count('order_id').alias('delay_count')
)

result.show()

+-----------+--------+---------+-------------+---------+
|customer_id|order_id|   status|customer_name|   region|
+-----------+--------+---------+-------------+---------+
|          1|     101|  delayed|     vaishali|  chennai|
|          2|     102|delivered|        rahul|bangalore|
|          3|     103|  delayed|        priya|hyderabad|
|          4|     104|  delayed|         arun|  chennai|
+-----------+--------+---------+-------------+---------+

+---------+-----------+
|   region|delay_count|
+---------+-----------+
|  chennai|          2|
|hyderabad|          1|
+---------+-----------+



In [11]:
result.toPandas().to_csv(
'delayed_region_output.csv',
index=False
)

# **Week - 4**

In [14]:
from pyspark.sql.functions import (col,to_date,datediff,when,current_date)
df=spark.read.csv('orders.csv',header=True,inferSchema=True)
df.show()

+-----------+-------------+-------------+---------+---------------+
|customer_id|customer_name|delivery_date|   status|          issue|
+-----------+-------------+-------------+---------+---------------+
|          1|     vaishali|   2026-05-01|  delayed|  late delivery|
|          2|        rahul|   2026-05-12|delivered|       no issue|
|          3|        priya|   2026-05-03|  delayed|damaged package|
|          4|         arun|   2026-05-08|  delayed|  late delivery|
+-----------+-------------+-------------+---------+---------------+



In [15]:
df_clean=df.withColumn(
'delivery_date',
to_date(col('delivery_date'),'yyyy-MM-dd'))
df_clean=df_clean.withColumn('delay_days',datediff(current_date(),col('delivery_date')))
df_clean=df_clean.withColumn('is_delayed',when(col('status')=='delayed',1).otherwise(0))
df_clean.show()

+-----------+-------------+-------------+---------+---------------+----------+----------+
|customer_id|customer_name|delivery_date|   status|          issue|delay_days|is_delayed|
+-----------+-------------+-------------+---------+---------------+----------+----------+
|          1|     vaishali|   2026-05-01|  delayed|  late delivery|        13|         1|
|          2|        rahul|   2026-05-12|delivered|       no issue|         2|         0|
|          3|        priya|   2026-05-03|  delayed|damaged package|        11|         1|
|          4|         arun|   2026-05-08|  delayed|  late delivery|         6|         1|
+-----------+-------------+-------------+---------+---------------+----------+----------+



In [16]:
top_customers=df_clean.groupBy('customer_name').sum('is_delayed')
top_customers.show()

+-------------+---------------+
|customer_name|sum(is_delayed)|
+-------------+---------------+
|        rahul|              0|
|         arun|              1|
|        priya|              1|
|     vaishali|              1|
+-------------+---------------+



In [17]:
df_clean.toPandas().to_csv('cleaned_orders.csv',index=False)